In [1]:
%load_ext autoreload
%autoreload 3 --print --log

In [3]:
import time
import asyncio
from rsim.core import Sensor
from rsim import Runtime, ProcessSensor, Map, Bundle, Camera, RobinW, allocate

In [68]:
class Counter(Sensor):

    def __init__(self):
        super().__init__(history=16)
        self.value = 0

    async def open(self):
        self.task(
            "generate",
            self.generate,
            hz=1,
            )

    async def generate(self):
        self.value += 1

        await self.publish(
            self.value,
            stamp_ns=time.time_ns(),
            clock="host:unix",
            )

In [51]:
async def run_runtime(*sensors):
    async with Runtime(*sensors):
        await asyncio.Event().wait()

In [70]:
counter = Counter()

runtime_task = asyncio.create_task(run_runtime(counter))

In [76]:
await counter.get()

Frame(data=308, stamp_ns=1790135522995236723, clock='host:unix', received_ns=1790135522995259444, sequence=308)

In [77]:
runtime_task.cancel()

try:
    await runtime_task
except asyncio.CancelledError:
    pass

In [78]:
asyncio.all_tasks()